# 🏆 Camada Gold – Data Marts Analíticos Olist

A **camada Gold** representa o **nível final da Arquitetura Medalhão**. Os dados, já limpos e estruturados na camada Silver, são transformados em **informação de negócio**: agregações, métricas, rankings e KPIs prontos para análise.

---

## 📊 Data Marts criados neste notebook

| Projeto | Tabela Gold | Descrição |
|:--|:--|:--|
| 1º Comercial | gold.fat_vendas_comercial | Receitas por Ano, Mês e Categoria |
| 2º Satisfação | gold.fat_avaliacoes_clientes | Avaliações por Categoria, Vendedor e Estado |

## ⚙️ Configuração e Leitura das Tabelas Silver

In [0]:
# Importação de bibliotecas
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Definição dos catálogos e schemas
catalogo = "medalhao"
silver_db_name = "silver"
gold_db_name = "gold"

print(f"Catálogo: {catalogo}")
print(f"Schema Silver (origem): {silver_db_name}")
print(f"Schema Gold (destino): {gold_db_name}")

Catálogo: medalhao
Schema Silver (origem): silver
Schema Gold (destino): gold


In [0]:
# ---------------------------------------------------------------
# Leitura das tabelas Silver que serão usadas para construção da Gold
# ---------------------------------------------------------------
fat_pedido_total_df = spark.table(f"{catalogo}.{silver_db_name}.fat_pedido_total")
fat_itens_pedidos_df = spark.table(f"{catalogo}.{silver_db_name}.fat_itens_pedidos")
fat_avaliacoes_df = spark.table(f"{catalogo}.{silver_db_name}.fat_avaliacoes_pedidos")
dim_produtos_df = spark.table(f"{catalogo}.{silver_db_name}.dim_produtos")
dim_vendedores_df = spark.table(f"{catalogo}.{silver_db_name}.dim_vendedores")
dim_categoria_df = spark.table(f"{catalogo}.{silver_db_name}.dim_categoria_produtos_traducao")

print("✅ Tabelas Silver carregadas com sucesso!")

✅ Tabelas Silver carregadas com sucesso!


---
# 🛒 1º Projeto: Visão Comercial e Volume de Produtos

A área comercial necessita de uma tabela agregada para acompanhar a **evolução das receitas** e dois recortes analíticos para identificar os produtos com **maior e menor saída** no histórico.

## Entrega 1 – Tabela Principal: gold.fat_vendas_comercial

**Granularidade:** Agrupamento por Ano, Mês e Categoria de Produto

**Métricas calculadas:**
- `total_pedidos` – contagem distinta de pedidos
- `qtd_itens_vendidos` – contagem absoluta de itens
- `receita_total_brl` – soma financeira em BRL
- `receita_total_usd` – soma financeira em USD
- `ticket_medio_brl` – receita / pedidos

In [0]:
# ---------------------------------------------------------------
# Tabela Gold: fat_vendas_comercial
# ---------------------------------------------------------------
# Etapa 1: une itens de pedidos com produtos para ter a categoria
itens_com_produto = (
    fat_itens_pedidos_df
    .join(dim_produtos_df.select("id_produto", "categoria_produto"), on="id_produto", how="left")
    .join(
        # Enriquece com nome em inglês da categoria via tabela de tradução
        dim_categoria_df,
        fat_itens_pedidos_df["id_produto"] == dim_categoria_df["nome_produto_pt"],
        how="left"
    )
)

# Etapa 2: une com pedido_total para ter data, status e valores em BRL/USD
base_comercial = (
    itens_com_produto
    .join(
        fat_pedido_total_df.select("id_pedido", "data_pedido", "valor_total_pago_brl", "valor_total_pago_usd"),
        on="id_pedido",
        how="inner"
    )
    # Extrai ano e mês da data do pedido para agrupamento temporal
    .withColumn("ano_venda", F.year(F.col("data_pedido")))
    .withColumn("mes_venda", F.month(F.col("data_pedido")))
)

# Etapa 3: agrega por Ano, Mês e Categoria
fat_vendas_comercial_df = (
    base_comercial
    .groupBy("ano_venda", "mes_venda", "categoria_produto")
    .agg(
        # Contagem distinta de pedidos únicos no período
        F.countDistinct("id_pedido").alias("total_pedidos"),
        # Contagem absoluta de linhas de itens vendidos
        F.count("id_item").alias("qtd_itens_vendidos"),
        # Soma da receita com arredondamento obrigatório de 2 casas decimais
        F.round(F.sum("preco_BRL"), 2).alias("receita_total_brl"),
        F.round(F.sum("preco_BRL") / F.first("valor_total_pago_brl") * F.first("valor_total_pago_usd"), 2).alias("receita_total_usd")
    )
    # Ticket médio: receita total dividida pela quantidade de pedidos únicos
    .withColumn(
        "ticket_medio_brl",
        F.round(F.col("receita_total_brl") / F.col("total_pedidos"), 2)
    )
    .orderBy("ano_venda", "mes_venda", "categoria_produto")
)

# Escrita na camada Gold em modo overwrite
(
    fat_vendas_comercial_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{gold_db_name}.fat_vendas_comercial")
)

print(f"✅ Tabela gold.fat_vendas_comercial criada com sucesso! ({fat_vendas_comercial_df.count()} registros)")
fat_vendas_comercial_df.display()

✅ Tabela gold.fat_vendas_comercial criada com sucesso! (1283 registros)


ano_venda,mes_venda,categoria_produto,total_pedidos,qtd_itens_vendidos,receita_total_brl,receita_total_usd,ticket_medio_brl
2016,9,beleza_saude,1,3,134.97,null,134.97
2016,9,moveis_decoracao,1,2,72.89,null,72.89
2016,9,telefonia,1,1,59.5,null,59.5
2016,10,null,2,2,65.89,null,32.95
2016,10,alimentos,1,1,79.9,null,79.9
2016,10,audio,2,2,156.99,null,78.5
2016,10,automotivo,11,12,1833.25,null,166.66
2016,10,bebes,11,14,1630.16,null,148.2
2016,10,beleza_saude,44,48,4552.51,null,103.47
2016,10,brinquedos,25,27,4465.09,null,178.6


## Entrega 2 – Rankings Comerciais (via display)

Rankings dos **Top 5 Produtos MAIS Vendidos** e **Top 5 Produtos MENOS Vendidos** com base na quantidade de itens vendidos.

In [0]:
# ---------------------------------------------------------------
# Ranking Comercial: Top 5 Produtos MAIS Vendidos
# Agrega por produto (id_produto) e ordena pela quantidade descendente
# ---------------------------------------------------------------
vendas_por_produto = (
    fat_itens_pedidos_df
    .join(dim_produtos_df.select("id_produto", "categoria_produto"), on="id_produto", how="left")
    .groupBy("id_produto", "categoria_produto")
    .agg(F.count("id_item").alias("quantidade_vendida"))
)

# Top 5 Produtos MAIS Vendidos
print("🏆 Top 5 Produtos MAIS Vendidos:")
top5_mais_vendidos_df = (
    vendas_por_produto
    .orderBy(F.col("quantidade_vendida").desc())
    .limit(5)
    .select(
        F.col("id_produto").alias("produto"),
        F.col("categoria_produto"),
        F.col("quantidade_vendida")
    )
)
top5_mais_vendidos_df.display()

🏆 Top 5 Produtos MAIS Vendidos:


produto,categoria_produto,quantidade_vendida
aca2eb7d00ea1a7b8ebd4e68314663af,moveis_decoracao,527
99a4788cb24856965c36a24e339b6058,cama_mesa_banho,488
422879e10f46682990de24d770e7f83d,ferramentas_jardim,484
389d119b48cf3043d311335e499d9c6b,ferramentas_jardim,392
368c6c730842d78016ad823897a372db,ferramentas_jardim,388


In [0]:
# ---------------------------------------------------------------
# Ranking Comercial: Top 5 Produtos MENOS Vendidos
# Mesma lógica, porém ordenação crescente por quantidade
# ---------------------------------------------------------------
print("📉 Top 5 Produtos MENOS Vendidos:")
top5_menos_vendidos_df = (
    vendas_por_produto
    .orderBy(F.col("quantidade_vendida").asc())
    .limit(5)
    .select(
        F.col("id_produto").alias("produto"),
        F.col("categoria_produto"),
        F.col("quantidade_vendida")
    )
)
top5_menos_vendidos_df.display()

📉 Top 5 Produtos MENOS Vendidos:


produto,categoria_produto,quantidade_vendida
e9f2586707fb45ea0f9997c54f585842,esporte_lazer,1
e1c0a39b7f806727ea5121c60035ed3c,informatica_acessorios,1
fa11ecd35f999783e96ac500532d9d45,cool_stuff,1
8d7ab3701456fdbfe2526636ce15327a,malas_acessorios,1
cdb8d3c880b6639a70764c6734e6bb69,beleza_saude,1


---
# ⭐ 2º Projeto: Satisfação de Clientes e Qualidade de Parceiros

A diretoria exige clareza sobre quais **categorias, produtos e vendedores** estão gerando as melhores e piores experiências de compra.

## Entrega 1 – Tabela Principal: gold.fat_avaliacoes_clientes

**Granularidade:** Agrupamento por Categoria do Produto, Nome do Vendedor e Estado do Vendedor

**Métricas calculadas:**
- `total_avaliacoes` – contagem absoluta
- `avaliacao_media` – média matemática das notas
- `total_avaliacoes_positivas` – notas >= 4
- `total_avaliacoes_negativas` – notas <= 2
- `percentual_satisfacao` – Positivas / Total * 100

In [0]:
# ---------------------------------------------------------------
# Tabela Gold: fat_avaliacoes_clientes
# ---------------------------------------------------------------
# Etapa 1: une avaliações com itens de pedidos para chegar ao produto e vendedor
avaliacoes_com_itens = (
    fat_avaliacoes_df
    .join(
        fat_itens_pedidos_df.select("id_pedido", "id_produto", "id_vendedor"),
        on="id_pedido",
        how="left"
    )
)

# Etapa 2: enriquece com dados de produto e vendedor
avaliacoes_completas = (
    avaliacoes_com_itens
    .join(dim_produtos_df.select("id_produto", "categoria_produto"), on="id_produto", how="left")
    .join(dim_vendedores_df.select("id_vendedor", "estado"), on="id_vendedor", how="left")
)

# Etapa 3: agrega por Categoria, Vendedor e Estado do Vendedor
fat_avaliacoes_clientes_df = (
    avaliacoes_completas
    .groupBy("categoria_produto", "id_vendedor", "estado")
    .agg(
        # Total absoluto de avaliações recebidas
        F.count("id_avaliacao").alias("total_avaliacoes"),
        # Média das notas arredondada para 2 casas decimais
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
        # Avaliações positivas: nota maior ou igual a 4
        F.sum(F.when(F.col("nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
        # Avaliações negativas: nota menor ou igual a 2
        F.sum(F.when(F.col("nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas")
    )
    # Percentual de satisfação: positivas dividido pelo total multiplicado por 100
    .withColumn(
        "percentual_satisfacao",
        F.round(
            F.col("total_avaliacoes_positivas") / F.col("total_avaliacoes") * 100,
            2
        )
    )
    .orderBy(F.col("avaliacao_media").desc(), F.col("total_avaliacoes").desc())
)

# Escrita na camada Gold em modo overwrite
(
    fat_avaliacoes_clientes_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{gold_db_name}.fat_avaliacoes_clientes")
)

print(f"✅ Tabela gold.fat_avaliacoes_clientes criada com sucesso! ({fat_avaliacoes_clientes_df.count()} registros)")
fat_avaliacoes_clientes_df.display()

✅ Tabela gold.fat_avaliacoes_clientes criada com sucesso! (6591 registros)


categoria_produto,id_vendedor,estado,total_avaliacoes,avaliacao_media,total_avaliacoes_positivas,total_avaliacoes_negativas,percentual_satisfacao
informatica_acessorios,a08692680c77d30a0b4280da5df01c5a,SP,17,5.0,17,0,100.0
alimentos_bebidas,5a93f3ab0ef4c84ed5e1b5dbf23978bc,SP,16,5.0,16,0,100.0
ferramentas_jardim,0b36063d5818f81ccb94b54adfaebbf5,SC,15,5.0,15,0,100.0
pet_shop,c8c1bea22194a4eefa2dc9a9fa89f536,SC,13,5.0,13,0,100.0
informatica_acessorios,2addf05f476d0637864454e93ba673d5,DF,12,5.0,12,0,100.0
fashion_bolsas_e_acessorios,48efc9d94a9834137efd9ea76b065a38,PR,12,5.0,12,0,100.0
sinalizacao_e_seguranca,2c9e548be18521d1c43cde1c582c6de8,SP,12,5.0,12,0,100.0
esporte_lazer,d13e50eaa47b4cbe9eb81465865d8cfc,SP,11,5.0,11,0,100.0
instrumentos_musicais,b2eecf5ea250510da76590ca79d60e5d,SP,11,5.0,11,0,100.0
construcao_ferramentas_construcao,53b0300ca793f9834cd69c0678d35ee8,SP,10,5.0,10,0,100.0


## Entrega 2 – Rankings de Qualidade (via display)

Critério de ordenação composto:
1. **Nota** (maior ou menor)
2. **Volume de avaliações** como critério de desempate (descendente)

In [0]:
# ---------------------------------------------------------------
# Preparação da base de avaliações por produto para os rankings
# Agrega métricas no nível de produto individual
# ---------------------------------------------------------------
avaliacoes_por_produto = (
    fat_avaliacoes_df
    .join(
        fat_itens_pedidos_df.select("id_pedido", "id_produto"),
        on="id_pedido",
        how="left"
    )
    .join(dim_produtos_df.select("id_produto", "categoria_produto"), on="id_produto", how="left")
    .groupBy("id_produto", "categoria_produto")
    .agg(
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
        F.count("id_avaliacao").alias("total_avaliacoes")
    )
)

In [0]:
# ---------------------------------------------------------------
# 1. Produto MAIS bem avaliado
# Critério: maior avaliação média, desempate por maior volume de avaliações
# ---------------------------------------------------------------
print("⭐ Produto MAIS bem avaliado:")
produto_mais_avaliado_df = (
    avaliacoes_por_produto
    .orderBy(
        F.col("avaliacao_media").desc(),
        F.col("total_avaliacoes").desc()  # Desempate por volume de avaliações
    )
    .limit(1)
)
produto_mais_avaliado_df.display()

⭐ Produto MAIS bem avaliado:


id_produto,categoria_produto,avaliacao_media,total_avaliacoes
37eb69aca8718e843d897aa7b82f462d,ferramentas_jardim,5.0,15


In [0]:
# ---------------------------------------------------------------
# 2. Produto MENOS bem avaliado
# Critério: menor avaliação média, desempate por maior volume de avaliações
# ---------------------------------------------------------------
print("👎 Produto MENOS bem avaliado:")
produto_menos_avaliado_df = (
    avaliacoes_por_produto
    .orderBy(
        F.col("avaliacao_media").asc(),
        F.col("total_avaliacoes").desc()  # Desempate por volume de avaliações
    )
    .limit(1)
)
produto_menos_avaliado_df.display()

👎 Produto MENOS bem avaliado:


id_produto,categoria_produto,avaliacao_media,total_avaliacoes
fb29f48bfea41db52e349454f433340e,informatica_acessorios,1.0,10


In [0]:
# ---------------------------------------------------------------
# Preparação da base de avaliações por vendedor para os rankings
# Agrega métricas no nível de vendedor individual
# ---------------------------------------------------------------
avaliacoes_por_vendedor = (
    fat_avaliacoes_df
    .join(
        fat_itens_pedidos_df.select("id_pedido", "id_vendedor"),
        on="id_pedido",
        how="left"
    )
    .join(dim_vendedores_df.select("id_vendedor", "estado"), on="id_vendedor", how="left")
    .groupBy("id_vendedor", "estado")
    .agg(
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
        F.count("id_avaliacao").alias("total_avaliacoes")
    )
)

In [0]:
# ---------------------------------------------------------------
# 3. Vendedor MAIS bem avaliado
# Critério: maior avaliação média, desempate por maior volume de avaliações
# ---------------------------------------------------------------
print("🥇 Vendedor MAIS bem avaliado:")
vendedor_mais_avaliado_df = (
    avaliacoes_por_vendedor
    .orderBy(
        F.col("avaliacao_media").desc(),
        F.col("total_avaliacoes").desc()  # Desempate por volume de avaliações
    )
    .limit(1)
)
vendedor_mais_avaliado_df.display()

🥇 Vendedor MAIS bem avaliado:


id_vendedor,estado,avaliacao_media,total_avaliacoes
48efc9d94a9834137efd9ea76b065a38,PR,5.0,34


In [0]:
# ---------------------------------------------------------------
# 4. Vendedor MENOS bem avaliado
# Critério: menor avaliação média, desempate por maior volume de avaliações
# ---------------------------------------------------------------
print("👎 Vendedor MENOS bem avaliado:")
vendedor_menos_avaliado_df = (
    avaliacoes_por_vendedor
    .orderBy(
        F.col("avaliacao_media").asc(),
        F.col("total_avaliacoes").desc()  # Desempate por volume de avaliações
    )
    .limit(1)
)
vendedor_menos_avaliado_df.display()

print("\n🎉 Notebook da Camada Gold finalizado com sucesso!")

👎 Vendedor MENOS bem avaliado:


id_vendedor,estado,avaliacao_media,total_avaliacoes
8d92f3ea807b89465643c219455e7369,SP,1.0,8



🎉 Notebook da Camada Gold finalizado com sucesso!


## ✅ Verificação Final – Tabelas Gold

In [0]:
# ---------------------------------------------------------------
# Verificação final: lista todas as tabelas criadas no schema Gold
# ---------------------------------------------------------------
print("📋 Tabelas criadas na camada Gold:")
spark.sql(f"SHOW TABLES IN {catalogo}.{gold_db_name}").display()

📋 Tabelas criadas na camada Gold:


database,tableName,isTemporary
gold,fat_avaliacoes_clientes,false
gold,fat_vendas_comercial,false
